
# **Fase 1: Perfilado Rápido (Clustering)**

- **Input:** Dataset `perfil_usuario_mx.csv` (823 registros).
- **Proceso:** K-Means para agrupar en 3 perfiles (Bajo, Medio, Alto Riesgo).
- **Output:** El "Perfil Base" del usuario al iniciar el chat.

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('mental_health_finaldata_1.csv')
# --- PASO 1: Mapeo de Variables Ordinales (Orden Lógico) ---
# 'Mood_Swings' representa tu escala de riesgo/estrés (Bajo, Medio, Alto).
# 'Age' y 'Days_Indoors' también tienen un orden que el modelo debe entender.
mood_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
age_mapping = {'16-20': 0, '20-25': 1, '25-30': 2, '30-Above': 3}
days_mapping = {
    'Go out Every day': 0, '1-14 days': 1, '15-30 days': 2, 
    '31-60 days': 3, 'More than 2 months': 4
}
# Mapeo para respuestas de intensidad (Yes=2, Maybe=1, No=0)
intensidad_map = {'Yes': 2, 'Maybe': 1, 'No': 0}

In [4]:
# Aplicamos las transformaciones
df['Mood_Swings'] = df['Mood_Swings'].map(mood_mapping)
df['Age'] = df['Age'].map(age_mapping)
df['Days_Indoors'] = df['Days_Indoors'].map(days_mapping)

In [5]:
# Columnas tipo Yes/Maybe/No
cols_intensidad = [
    'Growing_Stress', 'Quarantine_Frustrations', 'Changes_Habits', 
    'Mental_Health_History', 'Weight_Change', 'Coping_Struggles', 
    'Work_Interest', 'Social_Weakness'
]
for col in cols_intensidad:
    df[col] = df[col].map(intensidad_map)

In [6]:
# --- PASO 2: One-Hot Encoding (Variables Nominales) ---
# Género y Ocupación no tienen un orden (un Estudiante no es "mayor" a un Ingeniero).
# Creamos columnas independientes para evitar sesgos jerárquicos.
df_final = pd.get_dummies(df, columns=['Gender', 'Occupation'], prefix=['Gen', 'Occ'])

# Convertimos booleanos resultantes a enteros (0 y 1)
df_final = df_final.astype(int)

In [7]:
# --- PASO 3: Verificación y Exportación ---
print(f"Dataset procesado con éxito. Dimensiones: {df_final.shape}")
df_final.to_csv('perfil_usuario_mx.csv', index=False)

Dataset procesado con éxito. Dimensiones: (824, 18)


### Perfilado para el clustering
#### Agrupación de variables:
- **PHQ-9 (Depresión):**
  - Columnas `Mood_Swings`, `Changes_Habits` (sueño/hambre) y `Work_Interest` (anhedonia).
  - En el DSM-5, la pérdida de interés y el cambio de humor son criterios mayores para un episodio depresivo.

- **GAD-7 (Ansiedad):**
  - Columnas `Growing_Stress` y `Social_Weakness`.
  - El GAD-7 mide la preocupación constante y la tensión, en el dataset original de mental_health_finaldata se reflejan en el estrés creciente y la dificultad para socializar.
- **DSM-5:**
  - Coping_Struggles: Valida la incapacidad de lidiar con el estrés, un criterio de gravedad del DSM-5.

- **Factores de Riesgo:**
  - La columna `Mental_Health_History` es una "bandera roja".
  - Clínicamente, alguien con antecedentes tiene un umbral de riesgo más alto ante los mismos síntomas.
### Objetivo y preguntas asociadas:
  - ¿Es posible identificar grupos de riesgo (bajo, medio, alto) basándose únicamente en la similitud semántica de las conversaciones?
  - #3. Agrupamiento de Perfiles de Usuario: Segmentar a los usuarios por severidad de síntomas sin etiquetas previas.

In [1]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [2]:
import pandas as pd
import joblib  

In [3]:


# --- PARTE 1: Entrenamiento y Configuración del "Cerebro" ---
df = pd.read_csv('perfil_usuario_mx.csv')

# Definir TODAS las columnas clínicas (Lista Extendida para evitar sesgos)
columnas_clinicas = [
    'Growing_Stress', 'Coping_Struggles', 'Work_Interest', 
    'Mood_Swings', 'Changes_Habits', 'Social_Weakness', 
    'Mental_Health_History', 'Quarantine_Frustrations', 'Weight_Change'
]

# Estandarización (Vital para eliminar sesgos de escala)
scaler = StandardScaler()
columnas_modelo = [col for col in df.columns if col != 'Cluster']
df_scaled = scaler.fit_transform(df[columnas_modelo])

# Entrenamiento K-Means
model_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = model_kmeans.fit_predict(df_scaled)

# Cálculo de la Calidad (Silhouette Score)
score = silhouette_score(df_scaled, df['Cluster'])
print(f"Silhouette Score: {score:.4f}")

# Identificación Dinámica de Gravedad (Libro de Reglas)
centros = df.groupby('Cluster').mean()
centros['Carga_Sintomas'] = centros[columnas_clinicas].sum(axis=1) # Usamos la lista extendida
centros.to_csv('cluster_centroids.csv')

# Guardar para el MVP
joblib.dump(model_kmeans, 'modelo_clustering_v1.pkl')
joblib.dump(scaler, 'scaler_clustering.pkl')

# Identificar los IDs de los clusters dinámicamente
ID_ROJO = centros['Carga_Sintomas'].idxmax()
ID_VERDE = centros['Carga_Sintomas'].idxmin()


Silhouette Score: 0.1150


<small>
Aunque los promedios de síntomas como Work_Interest presentan una varianza baja (comorbilidad), el modelo K-Means identificó perfiles de riesgo distintos basados en el binomio Aislamiento-Edad. El Cluster 2 se identifica como Riesgo Alto no solo por el síntoma, sino por la combinación de juventud, falta de herramientas de afrontamiento y debilidad social, factores que disparan la necesidad de un Recall preventivo superior.
</small>


### 1. Escalas de las variables

| Variable                           | Rango | ¿Qué significa el valor?                                                   |
|------------------------------------|-------|----------------------------------------------------------------------------|
| Aislamiento (Days_Indoors)         | 0 a 4 | 0: Sale diario. 1: 1-14 días. 2: 15-30 días. 3: 31-60 días. 4: Más de 2 meses. |
| Síntomas (Estrés, Humor, etc.)     | 0 a 2 | 0: No / Bajo. 1: Tal vez / Moderado. 2: Sí / Alto.                          |
| Historial (historial)              | 0 a 2 | 0: No. 1: No sabe. 2: Sí.                                                  |
| Categoría de Edad (age_cat)        | 1 a 5 | 1: 16-20. 2: 21-25. 3: 26-30... (y así sucesivamente).                     |

### 2. Puntuación

- **puntuacion_real** = (estrés + interés + afrontamiento + humor + hábitos + debilidad_social)
- Cada una de estas 6 variables llega máximo a 2 → el tope de tu puntuación es **12**.
- Interpretación:
  - **6 o más** → Alerta clínica clara (**Riesgo Alto**).
  - **3 a 5** → **Riesgo Moderado**.
  - **0 a 2** → **Riesgo Bajo**.
- Nota: Si le pones un **4** al estrés, el *StandardScaler* lo detectará como un **outlier** (valor fuera de lo común) y podría:
  - Lanzar a ese usuario a un cluster totalmente distinto.
  - O incluso provocar un error de predicción.

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib

# --- 1. ENTRENAMIENTO Y CONFIGURACIÓN ---
df = pd.read_csv('perfil_usuario_mx.csv')
columnas_modelo = [col for col in df.columns if col != 'Cluster']

# Definimos las variables clínicas (DSM-5 / PHQ-9 / GAD-7)
cols_clinicas = [
    'Growing_Stress', 'Coping_Struggles', 'Work_Interest', 
    'Mood_Swings', 'Changes_Habits', 'Social_Weakness', 
    'Mental_Health_History', 'Weight_Change'
]

# Aplicamos Ponderación (Peso 3.0) para que los síntomas manden sobre la edad/trabajo
df_weighted = df[columnas_modelo].copy()
for col in cols_clinicas:
    df_weighted[col] = df_weighted[col] * 3.0

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_weighted)

model_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = model_kmeans.fit_predict(df_scaled)

# Identificación Dinámica de Clusters
centros_reales = df.groupby('Cluster').mean()
centros_reales['Carga'] = centros_reales[cols_clinicas].sum(axis=1)
orden = centros_reales['Carga'].sort_values().index
ID_VERDE, ID_AMARILLO, ID_ROJO = orden[0], orden[1], orden[2]

# --- 2. FUNCIÓN DEL PUENTE CLÍNICO CON CAPA DE DECISIÓN ---
def puente_clinico_definitivo(datos_usuario):
    """
    Combina K-Means con reglas clínicas para un discernimiento exacto.
    """
    cols = [
        'Age', 'Days_Indoors', 'Growing_Stress', 'Quarantine_Frustrations',
        'Changes_Habits', 'Mental_Health_History', 'Weight_Change',
        'Mood_Swings', 'Coping_Struggles', 'Work_Interest', 'Social_Weakness',
        'Gen_Female', 'Gen_Male', 'Occ_Business', 'Occ_Corporate',
        'Occ_Housewife', 'Occ_Others', 'Occ_Student'
    ]
    
    # Crear registro (Valor base neutral 0.5)
    reg = {c: 0.5 for c in cols}
    reg['Age'] = datos_usuario.get('age_cat', 1)
    reg['Gen_Female'] = 1 if datos_usuario['genero'] == 'femenino' else 0
    reg['Gen_Male'] = 1 if datos_usuario['genero'] == 'masculino' else 0
    reg['Days_Indoors'] = datos_usuario['aislamiento']
    reg['Growing_Stress'] = datos_usuario['estres']
    reg['Work_Interest'] = datos_usuario['interes']
    reg['Coping_Struggles'] = datos_usuario['afrontamiento']
    reg['Social_Weakness'] = datos_usuario['debilidad_social']
    reg['Mental_Health_History'] = datos_usuario['historial']
    reg['Mood_Swings'] = datos_usuario['humor']
    reg['Changes_Habits'] = datos_usuario['habitos']
    # Mapeo de variables físicas y emocionales restantes
    reg['Weight_Change'] = datos_usuario.get('peso', 1) # Cambios de peso/apetito
    reg['Quarantine_Frustrations'] = datos_usuario.get('irritabilidad', 1) # Irritabilidad ante el entorno
    
    for o in ['Occ_Business', 'Occ_Corporate', 'Occ_Housewife', 'Occ_Others', 'Occ_Student']:
        reg[o] = 0
    reg[f"Occ_{datos_usuario['ocupacion'].capitalize()}"] = 1

    # Inferencia de IA (K-Means)
    df_in = pd.DataFrame([reg])[cols]
    df_in_weighted = df_in.copy()
    for c in cols_clinicas: df_in_weighted[c] = df_in_weighted[c] * 3.0
    
    df_in_scaled = scaler.transform(df_in_weighted)
    cluster_ia = model_kmeans.predict(df_in_scaled)[0]
    
    # --- CAPA DE DECISIÓN CLÍNICA (Suma de Síntomas Reales) ---
    # Calculamos cuántos síntomas graves tiene el usuario (0 a 2 cada uno)
    puntuacion_real = (datos_usuario['estres'] + datos_usuario['interes'] + 
                       datos_usuario['afrontamiento'] + datos_usuario['humor'] +
                       datos_usuario['habitos'] + datos_usuario['debilidad_social'])
    
    # 3. Lógica de Discernimiento Final
    if cluster_ia == ID_ROJO:
        # Si la IA dice Rojo pero los síntomas son muy pocos, bajamos a Moderado
        if puntuacion_real <= 3: return "🟡 RIESGO MODERADO"
        return "🔴 RIESGO ALTO"
    
    if cluster_ia == ID_VERDE:
        # Si la IA dice Verde pero el usuario tiene historial o síntomas medios (Caso Adulto)
        if puntuacion_real >= 5 or datos_usuario['historial'] == 2:
            return "🔴 RIESGO ALTO (Ajuste por Gravedad Clínica)"
        if puntuacion_real >= 3:
            return "🟡 RIESGO MODERADO"
        return "🟢 RIESGO BAJO"
    
    return "🟡 RIESGO MODERADO"

# --- 3. TESTEO DE LOS 4 CASOS CRÍTICOS ---
casos_validación = [
    {"n": "1. Extremo (Grave)", "d": {'genero': 'femenino', 'aislamiento': 4, 'ocupacion': 'student', 'estres': 2, 'interes': 2, 'afrontamiento': 2, 'debilidad_social': 2, 'historial': 2, 'humor': 2, 'habitos': 2}},
    {"n": "2. Adulto en Crisis", "d": {'genero': 'masculino', 'age_cat': 3, 'aislamiento': 2, 'ocupacion': 'corporate', 'estres': 2, 'interes': 2, 'afrontamiento': 2, 'debilidad_social': 1, 'historial': 2, 'humor': 2, 'habitos': 2}},
    {"n": "3. Estudiante con Estrés", "d": {'genero': 'femenino', 'aislamiento': 0, 'ocupacion': 'student', 'estres': 1, 'interes': 0, 'afrontamiento': 1, 'debilidad_social': 0, 'historial': 0, 'humor': 0, 'habitos': 1}},
    {"n": "4. Persona Estable", "d": {'genero': 'masculino', 'aislamiento': 0, 'ocupacion': 'others', 'estres': 0, 'interes': 0, 'afrontamiento': 0, 'debilidad_social': 0, 'historial': 0, 'humor': 0, 'habitos': 0}}
]

print("\n--- RESULTADOS FINALES DE VALIDACIÓN (MVP ABRIL) ---")
for c in casos_validación:
    print(f"{c['n']} -> {puente_clinico_definitivo(c['d'])}")


--- RESULTADOS FINALES DE VALIDACIÓN (MVP ABRIL) ---
1. Extremo (Grave) -> 🔴 RIESGO ALTO
2. Adulto en Crisis -> 🔴 RIESGO ALTO (Ajuste por Gravedad Clínica)
3. Estudiante con Estrés -> 🟡 RIESGO MODERADO
4. Persona Estable -> 🟢 RIESGO BAJO


# **Fase 2: Procesamiento de Lenguaje (NLP)**

- **Input:** Dataset curado en español + SMOTE.
- **Proceso:** Fine-tuning de **RoBERTuito** con **QLoRA** (para que sea rápido).
- **Output:** Clasificación de Afecto (Neg/Pos/Neu) y Extracción de Síntomas (NER).

##### LIMPIEZA DE DATOS